# exp078 compact surface longtail gate inference

## Contents

1. Setup and configuration
2. Input prediction sources
3. Apply fixed gate policy
4. Metrics and submission

## 1. Setup and configuration

In [ ]:
from pathlib import Path
import json

import pandas as pd

from settings import ExperimentPaths, load_config
from exp063_full_replay_reproducibility_guard import run_compact_surface_longtail_inference

paths = ExperimentPaths()
config = load_config()
paths.artifacts_dir.mkdir(parents=True, exist_ok=True)

inference_config = config["inference"]
print(json.dumps({
    "experiment": config["experiment"]["name"],
    "parent": config["lineage"]["parent"],
    "compact_parent": config["lineage"]["compact_parent"],
    "inference": inference_config,
}, indent=2))

## 2. Input prediction sources

In [ ]:
base_predictions_path = config["data"].get("exp073_inference_predictions_local")
compact_predictions_path = config["data"].get("exp075_inference_predictions_local")
policy = inference_config.get("selected_policy", "tail_rank_ge1000_diff_p50_w005")

print("base:", base_predictions_path)
print("compact:", compact_predictions_path)
print("policy:", policy)
print("sample:", paths.sample_submission_path)

## 3. Apply fixed gate policy

In [ ]:
submission_path = paths.experiment_dir / "submission.csv"
summary = run_compact_surface_longtail_inference(
    output_dir=paths.artifacts_dir,
    submission_path=submission_path,
    sample_submission_path=paths.sample_submission_path,
    base_predictions_path=base_predictions_path,
    compact_predictions_path=compact_predictions_path,
    policy=policy,
    mode_name=inference_config.get("selected_mode", "gpu_repro_guard_dp_threads8"),
    model_name=inference_config.get("selected_model", "lgb_mean"),
    submission_target_column=config["data"].get("submission_target_column", "tvt"),
)
summary

## 4. Metrics and submission

In [ ]:
metrics_path = paths.artifacts_dir / "exp078_compact_surface_longtail_gate_inference_metrics.csv"
predictions_path = paths.artifacts_dir / "exp078_compact_surface_longtail_gate_inference_test_predictions.csv.gz"

metrics = pd.read_csv(metrics_path)
display(metrics)

submission = pd.read_csv(paths.experiment_dir / "submission.csv")
print(submission.shape)
display(submission.head())
print("predictions:", predictions_path)
print("submission:", paths.experiment_dir / "submission.csv")